Se carga la base de datos

Se trabaja con el módulo auxiliar process_data, el cual define una carga personalizada para estos ejemplos de ataques a modelos fedrados. Para más detalles sobre como efectuar la carga de datos con Flex ir a la documentación correspondiente.

Este módulo permite la carga de dataset de procesamiento de imágenes como: Mnist, Fmnist, Cifar10 y Cifar100. Además del dataset tabular nursery.

In [13]:
from process_data import *
from copy import deepcopy

flex_data, server_id = load_and_preprocess_horizontal(dataname="mnist", trasnform=False, nodes=5)

adv_data = flex_data[server_id]

def dict_data(data):
    x,y = data.to_numpy()
    mean = np.mean(x)
    std = np.std(x)
    data = {"x": np.expand_dims(x, axis = 1), "y": y.astype(np.int64), "mean": mean, "std": std}
    return data

adv_data_dict = dict_data(adv_data)

torch.Size([60000, 28, 28])


A continuación, se define la arquitectura de los modelos locales de los clientes. Para el presente ejemplo se trabaja con modelos neuronales de pytorch.

Se utiliza el módulo networks_models, quien contiene una serie de modelos neuronales auxiliares de pytorch, para el trabajo con las bases de datos anteriormente mencionadas. Además se utiliza el módulo auxiliar networks_execution, que define la ejecución del entrenamiento y otros detalles de estos modelos.

Para establecer un modelo personalizado, ir a la documentación de Flex.

In [14]:
from networks_models import *
from networks_execution import *
from flex.pool import init_server_model
from flex.model import FlexModel

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

net_config = ExecutionNetwork()

@init_server_model
def build_server_model():
    server_flex_model = FlexModel()

    criterion, model, optimizer = net_config.for_fd_server_model_config()

    server_flex_model["model"] = model.to(device)
    # Required to store this for later stages of the FL training process
    server_flex_model["criterion"] = criterion
    server_flex_model["optimizer_func"] = optimizer
    server_flex_model["optimizer_kwargs"] = {}

    return server_flex_model

Se define la arquitectura del modelo federado

In [15]:
from flex.pool import FlexPool
clients = 1

pool = FlexPool.client_server_pool(
        fed_dataset= flex_data, server_id=server_id, init_func = build_server_model
    )

#selected_test_clients_pool = pool.clients.select(clients)
#selected_test_clients = selected_test_clients_pool.clients

Se define la función para desplegar el modelo global en cada cliente

In [16]:
from flex.pool.decorators import (  # noqa: E402
    deploy_server_model,
)

@deploy_server_model
def deploy_serv(server_flex_model: FlexModel): 

    new_model = deepcopy(server_flex_model)

    return new_model

#pool.servers.map(deploy_serv, selected_test_clients)

Se define la ronda de entrenamiento local de un cliente, empleando el módulo networks_execution

In [17]:
def train(client_flex_model: FlexModel, client_data: Dataset):

    train_dataset = client_data.to_torchvision_dataset(transform = mnist_transform())
    client_dataloader = DataLoader(train_dataset, batch_size = 256)

    model = client_flex_model["model"]
    model = model.to(device)

    client_flex_model["previous_model"] = deepcopy(model)
    optimizer = client_flex_model["optimizer_func"]
    criterion = client_flex_model["criterion"]

    net_config.train_network(local_epochs = 1, criterion = criterion, optimizer = optimizer,
                            momentum = 0.9, lr = 0.005, trainloader = client_dataloader, testloader= None, 
                            model=model)
    
    return client_flex_model

#selected_test_clients.map(train)

Se define el atauqe de inferencia de propiedad, para ello es necesario mantener un control de las rondas de entrenaiento y los clientes que participan en cada ronda

In [18]:
from attack.my_models_attacks.property_inference import Passive_property_inference
import traceback
from flexclash.model import model_inference_known_a_model_information, model_inference_known_agregator_information

criterion, model, optimizer = net_config.for_fd_server_model_config()

round = 0
client_index = dict()
previus_dict = dict()

def find_client_id(index):
    for key, value in client_index.items():
        if value == index:
            return key
    return None

print(optimizer.__class__.__name__)
passive_attack = Passive_property_inference(data = adv_data_dict, criterion = criterion, model_server = model,
                                            batch=64, lr=0.001, optimizer = optimizer.__class__.__name__ , grad_clip = 2,
                                            local_upt_num = 5, type_classifier = "lr")

@model_inference_known_agregator_information
def inferencer(aggregated_weights_as_list: list): #aggregated_weights_as_list: list Esto va para el agregador
    try:
        new_list = deepcopy(aggregated_weights_as_list)
        for client_id in range(len(new_list)):
            id = find_client_id(client_id)
            passive_attack.collect_updates(new_list[client_id], round, id)#Considerar que aqui ya entra la resta entre los paramtetros actuales y los previos
    except Exception as e:
        print(f"Error: {e.__class__.__name__} - {e}")
        tb = traceback.format_exc()
        print(tb)

    return aggregated_weights_as_list

@model_inference_known_a_model_information
def PIA_inference_server(server_model: FlexModel):
    passive_attack.act_server_model(server_model["model"])
    passive_attack.get_data_for_dataset_prop_classifier()#Si con esto no se actualiza el diccionario, hacerlo una variable global
    return server_model

def train_and_infer():
    passive_attack.train_property_classifier()
    result = passive_attack.infer_collected()
    for k in result.keys():
        print("Round:", k)
        for client_id in result[k].keys():
            print("Cliente:", client_id, "Resultado", result[k][client_id])
        print("-----------------------------------------------------------------------------------")

No module named 'federatedscope' in `federatedscope.contrib.optimizer`, some modules are not available.


SGD


Se efectúa la agregación del modelo federado

In [19]:
from flex.pool import collect_client_diff_weights_pt
from flex.pool import fed_avg
from flex.pool import set_aggregated_diff_weights_pt


#pool.aggregators.map(collect_client_diff_weights_pt, selected_test_clients)
#pool.aggregators.map(fed_avg)
#pool.aggregators.map(set_aggregated_diff_weights_pt, pool.servers)

Se evalúa el modelo federado

In [20]:
def evaluate_global_model(server_flex_model: FlexModel, test_data: Dataset):#falta poner esto
    model = server_flex_model["model"]
    model.eval()
    test_loss = 0
    test_acc = 0
    total_count = 0
    model = model.to(device)
    criterion = server_flex_model["criterion"]
    # get test data as a torchvision object
    test_dataset = test_data.to_torchvision_dataset(transform = mnist_transform())
    test_dataloader = DataLoader(
        test_dataset, batch_size=256, shuffle=True, pin_memory=False
    )
    losses = []
    with torch.no_grad():
        for data, target in tqdm(test_dataloader):
            total_count += target.size(0)
            data, target = data.to(device), target.to(device)
            output = model(data)
            losses.append(criterion(output, target).item())
            pred = output.data.max(1, keepdim=True)[1]
            test_acc += pred.eq(target.data.view_as(pred)).long().cpu().sum().item()

    test_loss = sum(losses) / len(losses)
    test_acc /= total_count
    return test_loss, test_acc

#metrics = pool.servers.map(evaluate_global_model)

#loss, acc = metrics[0]
#print(f"Server: Test acc: {acc:.4f}, test loss: {loss:.4f}")

Para limpiar los modelos en memoria. Opcional

In [21]:
def clean_up_models(client_model: FlexModel, _):
    import gc

    client_model.clear()
    gc.collect()

Se definen las rondas de entrenamiento del modelo federado

In [22]:
def train_n_rounds(n_rounds, clients_per_round=20):
    pool = FlexPool.client_server_pool(
        fed_dataset= flex_data, server_id=server_id, init_func=build_server_model
    )
    global round
    global client_index    
    for i in range(n_rounds):
        print(f"\nRunning round: {i+1} of {n_rounds}")
        selected_clients_pool = pool.clients.select(clients_per_round)
        selected_clients = selected_clients_pool.clients
        pos_c_list=0
        client_index = dict()
        for i in selected_clients.actor_ids:
            print("Cliente id:", i)
            client_index[i] = pos_c_list
            pos_c_list+=1
        pool.servers.map(deploy_serv, selected_clients)
        selected_clients.map(train)
        pool.aggregators.map(collect_client_diff_weights_pt, selected_clients)
        pool.aggregators.map(inferencer)
        pool.aggregators.map(fed_avg)
        pool.aggregators.map(set_aggregated_diff_weights_pt, pool.servers)
        metrics = pool.servers.map(evaluate_global_model)
        pool.servers.map(PIA_inference_server)
        selected_clients.map(clean_up_models)
        loss, acc = metrics[0]
        print(f"Server: Test acc: {acc:.4f}, test loss: {loss:.4f}")
        round+=1
    train_and_infer()

In [23]:
train_n_rounds(4, clients_per_round = 2)


Running round: 1 of 4
Cliente id: 3
Cliente id: 4


100%|██████████| 47/47 [00:06<00:00,  6.84it/s]


Después de la modif antes de fedavg tensor(0.5048)



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\Adrian\PycharmProjects\pythonProject\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 739,

AttributeError: _ARRAY_API not found

...

AttributeError: _ARRAY_API not found

ImportError: numpy.core._multiarray_umath failed to import

100%|██████████| 40/40 [00:03<00:00, 10.61it/s]


The decorated function: PIA_inference_server had an error during execution


KeyboardInterrupt: 